In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 5 — Clasificación con Titanic
# Machine Learning con Python y scikit-learn
# ---------------------------------------------------------------

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
)

import sklearn
print('scikit-learn:', sklearn.__version__)  # anota la versión
print('pandas:      ', pd.__version__)        # anota la versión

In [ ]:
# --- Cargar Titanic desde seaborn ------------------------------
titanic = sns.load_dataset('titanic')

print('Forma del dataset:', titanic.shape)
print('\nColumnas y tipos:')
print(titanic.dtypes)
print('\nValores faltantes por columna:')
faltantes = titanic.isnull().sum()
print(faltantes[faltantes > 0])

In [ ]:
# --- Distribución del target -----------------------------------
print('Distribución de supervivencia:')
print(titanic['survived'].value_counts(normalize=True).round(3))

# --- Tasa de supervivencia por sexo y clase --------------------
tabla = titanic.groupby(['sex', 'pclass'])['survived'].mean()
print('\nTasa de supervivencia por sexo y clase:')
print(tabla.round(3).to_string())

In [ ]:
# --- Imputar edad con la mediana ------------------------------
# Nota: median() devuelve NaN si todos los valores son NaN.
# En este dataset hay 177 NaN de 891 — la mediana existe.
# Si todos fueran NaN, usa fillna(0) o un valor por defecto.
mediana_edad = titanic['age'].median()
titanic = titanic.copy()  # evita SettingWithCopyWarning
titanic['age'] = titanic['age'].fillna(mediana_edad)

print(f'Mediana de edad: {mediana_edad}')
print(f'NaN en age después: {titanic["age"].isnull().sum()}')

# --- Codificar sexo: female=1, male=0 -------------------------
titanic['sex_cod'] = titanic['sex'].map({'female': 1, 'male': 0})

print('\nVerificación de codificación:')
print(titanic[['sex', 'sex_cod']].drop_duplicates())

In [ ]:
# --- Selección de features y target ----------------------------
features = ['age', 'sex_cod', 'pclass']
X = titanic[features].values
y = titanic['survived'].values

# --- División 80/20 train/test ---------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # mantiene la proporción 62/38 en ambos conjuntos
)

print('Entrenamiento:', X_train.shape)
print('Test:         ', X_test.shape)
print('Proporción en train:', y_train.mean().round(3))
print('Proporción en test: ', y_test.mean().round(3))

In [ ]:
# --- Escalado --------------------------------------------------
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)   # nunca fit_transform en test

# --- Entrenar LogisticRegression --------------------------------
# lbfgs es el solver por defecto — adecuado para datasets pequeños
model = LogisticRegression(random_state=42)

# try:
model.fit(X_train_s, y_train)
# except Exception as e:
#     print(f'Error en entrenamiento: {e}')

# --- Ver coeficientes ------------------------------------------
coefs = pd.Series(model.coef_[0], index=features)
print('Intercepto:', round(model.intercept_[0], 4))
print('\nCoeficientes:')
print(coefs.round(4).to_string())

In [ ]:
# --- Predicciones -----------------------------------------------
y_pred      = model.predict(X_test_s)
y_pred_prob = model.predict_proba(X_test_s)[:, 1]  # P(sobrevivir)

# --- Exactitud --------------------------------------------------
acc = accuracy_score(y_test, y_pred)
print(f'Exactitud: {acc:.4f}  ({acc * 100:.1f}%)')

# --- Matriz de confusión ----------------------------------------
cm = confusion_matrix(y_test, y_pred)
print('\nMatriz de confusión:')
print(pd.DataFrame(
    cm,
    index=['Real: No (0)', 'Real: Sí (1)'],
    columns=['Pred: No (0)', 'Pred: Sí (1)']
))

# --- Reporte completo -------------------------------------------
print('\nReporte de clasificación:')
print(classification_report(y_test, y_pred,
    target_names=['No sobrevivió', 'Sobrevivió']))

In [ ]:
# --- Curva ROC --------------------------------------------------
fpr, tpr, umbrales = roc_curve(y_test, y_pred_prob)
auc = roc_auc_score(y_test, y_pred_prob)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='steelblue', lw=2,
        label=f'Regresión Logística (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'r--', lw=1.5,
        label='Clasificador aleatorio (AUC = 0.500)')
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos (Recall)')
ax.set_title('Curva ROC — Titanic')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(f'AUC-ROC: {auc:.4f}')